In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
import warnings
import subprocess
import time
import os
import time
import sqlite3
import glob
import time

warnings.filterwarnings('ignore')

caminho_absoluto = r'C:\Users\lucas\repositorios\mestrado_luedsbr\SRC\analista_cenario'
sys.path.append(caminho_absoluto)

try:
    from analistaContigencia import *
except ModuleNotFoundError as e:
    print(f" Erro: {e}")

# =============================================================================
# CONFIGURAÇÃO
# =============================================================================

TOTAL_CEN = 5
script_julia = r"C:\Users\lucas\repositorios\mestrado_luedsbr\SRC\modelos_matematicos\fluxPotContigencia.jl"

In [2]:
# =============================================================================
# 1. EXECUÇÃO DOS CENÁRIOS JULIA (SIMPLES)
# =============================================================================

def executar_cenarios_julia(total_cenarios, script_path):
    print(f"🎯 Executando {total_cenarios} cenários Julia...")
    
    tempo_inicio = time.time()
    sucessos = 0
    
    for i in range(total_cenarios):
        resultado = subprocess.run(
            ['julia', script_path],
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='ignore'  # IGNORA ERROS DE UNICODE
        )
        
        if resultado.returncode == 0:
            print("█", end="", flush=True)
            sucessos += 1
        else:
            print("❌", end="", flush=True)
            # MOSTRA APENAS O ERRO PRINCIPAL
            if resultado.stderr:
                linhas = resultado.stderr.split('\n')
                for linha in linhas:
                    if 'Error' in linha or 'ERROR' in linha:
                        print(f"\nERRO: {linha}")
                        break
    
    tempo_total = time.time() - tempo_inicio
    print(f"\n✅ Concluído! {sucessos}/{total_cenarios} sucessos")
    return tempo_total

# EXECUTAR
tempo_execucao = executar_cenarios_julia(TOTAL_CEN, script_julia)

🎯 Executando 5 cenários Julia...
█████
✅ Concluído! 5/5 sucessos


In [3]:
# =============================================================================
# 2. CONSOLIDAR RESULTADOS
# =============================================================================

def consolidar_resultados():
    print("📊 Consolidando resultados...")
    
    # Encontrar todos os bancos de dados
    bancos = glob.glob("resultados_opf_contingencias*.db")
    print(f"Encontrados {len(bancos)} bancos de dados")
    
    # Criar banco consolidado
    conn_destino = sqlite3.connect("resultados_consolidados.db")
    
    total_registros = 0
    for banco in bancos:
        try:
            conn_origem = sqlite3.connect(banco)
            
            # Copiar todas as tabelas
            for tabela in ['execucoes', 'contingencias', 'geradores_contingencia', 
                          'barras_contingencia', 'linhas_contingencia']:
                try:
                    df = pd.read_sql_query(f"SELECT * FROM {tabela}", conn_origem)
                    if len(df) > 0:
                        df.to_sql(tabela, conn_destino, if_exists='append', index=False)
                        total_registros += len(df)
                except:
                    continue
            
            conn_origem.close()
        except:
            continue
    
    conn_destino.close()
    print(f"✅ Consolidados {total_registros} registros")
    return total_registros

# CONSOLIDAR
total_registros = consolidar_resultados()

📊 Consolidando resultados...
Encontrados 1 bancos de dados
✅ Consolidados 305 registros


In [4]:
# =============================================================================
# 3. CARREGAR DADOS PARA ANÁLISE
# =============================================================================

def carregar_dados_consolidados():
    print("📈 Carregando dados consolidados...")
    
    conn = sqlite3.connect("resultados_consolidados.db")
    
    dados = {}
    tabelas = ['execucoes', 'contingencias', 'geradores_contingencia', 
               'barras_contingencia', 'linhas_contingencia']
    
    for tabela in tabelas:
        try:
            dados[tabela] = pd.read_sql_query(f"SELECT * FROM {tabela}", conn)
            print(f"✅ {tabela}: {len(dados[tabela])} registros")
        except:
            dados[tabela] = pd.DataFrame()
            print(f"❌ {tabela}: não disponível")
    
    conn.close()
    return dados

# CARREGAR DADOS
dados = carregar_dados_consolidados()

📈 Carregando dados consolidados...
✅ execucoes: 5 registros
✅ contingencias: 25 registros
✅ geradores_contingencia: 125 registros
✅ barras_contingencia: 75 registros
✅ linhas_contingencia: 75 registros


In [5]:
# =============================================================================
# 4. ANÁLISE E GRÁFICOS
# =============================================================================

import plotly.express as px
import plotly.graph_objects as go
import numpy as np

def plotar_evolucao_geracao_curtailment(dados):
    """Gráfico da evolução da geração eólica e curtailment"""
    if 'contingencias' not in dados or len(dados['contingencias']) == 0:
        print("❌ Dados insuficientes para gráfico")
        return
    
    df = dados['contingencias']
    
    # Agrupar por execução
    stats_por_execucao = df.groupby('id_execucao').agg({
        'total_carga_pu': 'mean',
        'total_curtailment_pu': 'mean',
        'total_deficit_pu': 'mean',
        'custo_total_usd_h': 'mean'
    }).reset_index()
    
    # Adicionar geração eólica (precisa dos dados dos geradores)
    if 'geradores_contingencia' in dados:
        geradores_gwd = dados['geradores_contingencia'][dados['geradores_contingencia']['tipo'] == 'GWD']
        geracao_por_execucao = geradores_gwd.groupby('id_execucao')['geracao_pu'].sum().reset_index()
        stats_por_execucao = stats_por_execucao.merge(geracao_por_execucao, on='id_execucao', how='left')
        stats_por_execucao['geracao_pu'] = stats_por_execucao['geracao_pu'].fillna(0)
    else:
        stats_por_execucao['geracao_pu'] = 0
    
    # Criar gráfico
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=stats_por_execucao.index, y=stats_por_execucao['geracao_pu'],
        name='Geração Eólica', line=dict(color='green', width=3)
    ))
    
    fig.add_trace(go.Scatter(
        x=stats_por_execucao.index, y=stats_por_execucao['total_carga_pu'],
        name='Demanda', line=dict(color='red', width=3, dash='dash')
    ))
    
    fig.add_trace(go.Scatter(
        x=stats_por_execucao.index, y=stats_por_execucao['total_curtailment_pu'],
        name='Curtailment', line=dict(color='orange', width=2)
    ))
    
    fig.update_layout(
        title='Evolução da Geração Eólica, Demanda e Curtailment',
        xaxis_title='Ordem dos Cenários',
        yaxis_title='Potência (pu)',
        height=500
    )
    
    fig.show()
    return stats_por_execucao

def plotar_distribuicao_custos(dados):
    """Histograma da distribuição de custos"""
    if 'contingencias' not in dados:
        return
    
    df = dados['contingencias']
    
    fig = px.histogram(df, x='custo_total_usd_h', 
                      title='Distribuição dos Custos Totais',
                      nbins=20)
    fig.show()

def plotar_curtailment_vs_demanda(dados):
    """Scatter plot: curtailment vs demanda"""
    if 'contingencias' not in dados:
        return
    
    df = dados['contingencias']
    
    fig = px.scatter(df, x='total_carga_pu', y='total_curtailment_pu',
                    title='Curtailment vs Demanda',
                    trendline='lowess')
    fig.show()

def analisar_rampas_simples(dados):
    """Análise simples das rampas entre cenários"""
    if 'contingencias' not in dados:
        return
    
    df = dados['contingencias']
    
    # Agrupar por execução e calcular totais
    execucoes_agrupadas = df.groupby('id_execucao').agg({
        'total_curtailment_pu': 'sum',
        'total_carga_pu': 'mean',
        'custo_total_usd_h': 'mean'
    }).reset_index()
    
    # Calcular variações entre execuções consecutivas
    rampas = []
    for i in range(1, len(execucoes_agrupadas)):
        anterior = execucoes_agrupadas.iloc[i-1]
        atual = execucoes_agrupadas.iloc[i]
        
        rampas.append({
            'transicao': f"{i}→{i+1}",
            'delta_curtailment': atual['total_curtailment_pu'] - anterior['total_curtailment_pu'],
            'delta_demanda': atual['total_carga_pu'] - anterior['total_carga_pu'],
            'delta_custo': atual['custo_total_usd_h'] - anterior['custo_total_usd_h']
        })
    
    df_rampas = pd.DataFrame(rampas)
    
    print("📊 Análise de Rampas:")
    print(df_rampas.round(4))
    
    # Gráfico das rampas
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        name='Variação Curtailment',
        x=df_rampas['transicao'],
        y=df_rampas['delta_curtailment'],
        marker_color='orange'
    ))
    
    fig.add_trace(go.Bar(
        name='Variação Demanda',
        x=df_rampas['transicao'],
        y=df_rampas['delta_demanda'],
        marker_color='red'
    ))
    
    fig.update_layout(
        title='Variações de Curtailment e Demanda entre Cenários',
        barmode='group'
    )
    
    fig.show()
    
    return df_rampas


In [6]:
# =============================================================================
# 5. EXECUTAR TODAS AS ANÁLISES
# =============================================================================

print("\n" + "="*60)
print("📊 GERANDO ANÁLISES E GRÁFICOS")
print("="*60)

# Gráfico 1: Evolução temporal
stats_evolucao = plotar_evolucao_geracao_curtailment(dados)

# Gráfico 2: Distribuição de custos
plotar_distribuicao_custos(dados)

# Gráfico 3: Relação curtailment vs demanda
plotar_curtailment_vs_demanda(dados)

# Análise 4: Rampas entre cenários
df_rampas = analisar_rampas_simples(dados)


📊 GERANDO ANÁLISES E GRÁFICOS


📊 Análise de Rampas:
  transicao  delta_curtailment  delta_demanda  delta_custo
0       1→2                0.0         0.2789    7893.6045
1       2→3                0.0        -0.1934  -12452.8978
2       3→4                0.0        -0.0297   11261.5572
3       4→5                0.0        -0.0175  -17028.5055


In [7]:
# =============================================================================
# 6. ESTATÍSTICAS RESUMO
# =============================================================================

def gerar_resumo_estatistico(dados):
    """Gerar resumo estatístico final"""
    print("\n" + "="*60)
    print("📈 RESUMO ESTATÍSTICO FINAL")
    print("="*60)
    
    if 'contingencias' in dados and len(dados['contingencias']) > 0:
        df = dados['contingencias']
        
        print(f"📊 Total de contingências analisadas: {len(df)}")
        print(f"🔢 Total de execuções únicas: {df['id_execucao'].nunique()}")
        
        print(f"💡 Demanda média: {df['total_carga_pu'].mean():.4f} pu")
        print(f"🌬️  Curtailment médio: {df['total_curtailment_pu'].mean():.4f} pu")
        print(f"⚡ Déficit médio: {df['total_deficit_pu'].mean():.4f} pu")
        print(f"💰 Custo médio: ${df['custo_total_usd_h'].mean():.2f}/h")
        
        # Contingências com déficit
        ctg_com_deficit = df[df['total_deficit_pu'] > 0.01]
        print(f"⚠️  Contingências com déficit: {len(ctg_com_deficit)}")
        
    if 'geradores_contingencia' in dados:
        geradores_gwd = dados['geradores_contingencia'][dados['geradores_contingencia']['tipo'] == 'GWD']
        if len(geradores_gwd) > 0:
            print(f"🌪️  Total de geradores eólicos: {geradores_gwd['id_gerador'].nunique()}")

# GERAR RESUMO
gerar_resumo_estatistico(dados)

print(f"\n✅ PROCESSO CONCLUÍDO!")
print(f"⏱️  Tempo de execução: {tempo_execucao:.1f} segundos")
print(f"📁 Registros consolidados: {total_registros}")
print(f"🎯 Cenários executados: {TOTAL_CEN}")


📈 RESUMO ESTATÍSTICO FINAL
📊 Total de contingências analisadas: 25
🔢 Total de execuções únicas: 5
💡 Demanda média: 0.9162 pu
🌬️  Curtailment médio: 0.0000 pu
⚡ Déficit médio: 0.3688 pu
💰 Custo médio: $74161.21/h
⚠️  Contingências com déficit: 15
🌪️  Total de geradores eólicos: 1

✅ PROCESSO CONCLUÍDO!
⏱️  Tempo de execução: 65.2 segundos
📁 Registros consolidados: 305
🎯 Cenários executados: 5
